# SiamViTRM — Fingerprint Verification (Colab)
Self-contained. Dataset downloaded from Hugging Face automatically.

In [4]:
!pip install -q datasets huggingface_hub Pillow scikit-learn torchvision
print("✓ Dependencies installed")


✓ Dependencies installed


In [5]:
from __future__ import annotations
import copy, csv, json, math, os, random, sys, time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR     = Path("/content/siamvitrm")
ARTIFACT_DIR = BASE_DIR / "artifacts"
TRM_DIR      = ARTIFACT_DIR / "trm"
FIG_DIR      = TRM_DIR / "figures"
DATA_DIR     = BASE_DIR / "dataset" / "FVC2004"
for d in [TRM_DIR, FIG_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE_TRM = (128, 128)
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

print(f"BASE_DIR : {BASE_DIR}")
print(f"TRM_DIR  : {TRM_DIR}")
print(f"Device   : {DEVICE}")


BASE_DIR : /content/siamvitrm
TRM_DIR  : /content/siamvitrm/artifacts/trm
Device   : cuda


## Model Definition

In [6]:
# ── Inlined from trm_model.py ─────────────────────────────────────────────────
import copy, math
from typing import Optional, Tuple

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return self.scale * x / x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()

class SwiGLU(nn.Module):
    def __init__(self, d):
        super().__init__()
        hidden = ((max(32, int(2/3*4*d)) + 31) // 32) * 32
        self.w1 = nn.Linear(d, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d, bias=False)
        self.w3 = nn.Linear(d, hidden, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class TRMLayer(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn  = nn.MultiheadAttention(d, n_heads, bias=False, batch_first=True)
        self.norm2 = RMSNorm(d)
        self.ffn   = SwiGLU(d)
    def forward(self, x):
        h = self.norm1(x); h, _ = self.attn(h, h, h, need_weights=False); x = x + h
        return x + self.ffn(self.norm2(x))

class FingerprintPatchEmbed(nn.Module):
    def __init__(self, img_size=128, patch_size=16, d_model=128):
        super().__init__()
        assert img_size % patch_size == 0
        self.patch_size = patch_size
        self.n_patches  = (img_size // patch_size) ** 2
        self.proj       = nn.Linear(patch_size * patch_size, d_model, bias=False)
        self.pos_embed  = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    def forward(self, x):
        B, C, H, W = x.shape; P = self.patch_size
        x = x.reshape(B, C, H//P, P, W//P, P).permute(0,2,4,3,5,1).reshape(B,-1,P*P*C)
        return self.proj(x) + self.pos_embed

class ViTRMEncoder(nn.Module):
    def __init__(self, d_model=128, n_heads=4, n_blocks=3, K=16, n_latent_steps=3, T_recursion=1):
        super().__init__()
        self.K = K; self.n_latent_steps = n_latent_steps; self.T_recursion = T_recursion
        self.block  = nn.Sequential(*[TRMLayer(d_model, n_heads) for _ in range(n_blocks)])
        self.y_init = nn.Parameter(torch.zeros(1, 1, d_model))
        self.z_init = nn.Parameter(torch.zeros(1, K, d_model))
        nn.init.trunc_normal_(self.y_init, std=0.02)
        nn.init.trunc_normal_(self.z_init, std=0.02)

    def _step_z(self, x, y, z):
        return self.block(torch.cat([x, y, z], dim=1))[:, -self.K:]
    def _step_y(self, y, z):
        return self.block(torch.cat([y, z], dim=1))[:, :1]
    def _cycle(self, x, y, z):
        for _ in range(self.n_latent_steps): z = self._step_z(x, y, z)
        return self._step_y(y, z), z
    def forward(self, x, y=None, z=None):
        B = x.size(0)
        if y is None: y = self.y_init.expand(B, -1, -1)
        if z is None: z = self.z_init.expand(B, -1, -1)
        if self.T_recursion > 1:
            with torch.no_grad():
                for _ in range(self.T_recursion - 1): y, z = self._cycle(x, y, z)
        y_grad, z_grad = self._cycle(x, y, z)
        return y_grad.detach(), z_grad.detach(), y_grad

class SiamViTRM(nn.Module):
    def __init__(self, img_size=128, patch_size=16, d_model=128, n_heads=4,
                 n_blocks=3, K=16, n_latent_steps=3, T_recursion=1):
        super().__init__()
        self.patch_embed = FingerprintPatchEmbed(img_size, patch_size, d_model)
        self.encoder     = ViTRMEncoder(d_model, n_heads, n_blocks, K, n_latent_steps, T_recursion)
        self.halt_head   = nn.Linear(d_model, 1, bias=False)
    def encode(self, img, y=None, z=None):
        return self.encoder(self.patch_embed(img), y, z)
    def forward(self, img1, img2, y1=None, z1=None, y2=None, z2=None):
        y1_det, z1_det, y1g = self.encode(img1, y1, z1)
        y2_det, z2_det, y2g = self.encode(img2, y2, z2)
        v1    = y1g.squeeze(1); v2 = y2g.squeeze(1)
        score = (F.cosine_similarity(v1, v2, dim=1) + 1.0) / 2.0
        q     = torch.sigmoid(self.halt_head((v1 + v2) / 2.0).squeeze(-1))
        return score, q, y1_det, z1_det, y2_det, z2_det, v1, v2

class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.eval_model = copy.deepcopy(model)
        self.eval_model.eval()
        for p in self.eval_model.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for ep, p in zip(self.eval_model.parameters(), model.parameters()):
            ep.data.mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✓ Model classes defined")


✓ Model classes defined


## Download FVC2004 from Hugging Face

In [7]:
from huggingface_hub import snapshot_download

print("Downloading FVC2004 from Hugging Face...")
snapshot_download(
    repo_id="tourmii/FVC2004",
    repo_type="dataset",
    local_dir=str(DATA_DIR),
)
print("✓ Downloaded. Contents:")
for p in sorted(DATA_DIR.iterdir()):
    print(f"  {p.name}/")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

✓ Downloaded. Contents:
  .cache/
  .gitattributes/
  Dbs/
  Doc/
  Src/


## Build Pair CSVs (finger-level split)

In [8]:
import itertools, re
from collections import defaultdict

FNAME_RE                  = re.compile(r'^(\d+)_(\d+)\.(bmp|png|tif|tiff|jpg|jpeg)$', re.IGNORECASE)
MAX_GENUINE_PAIRS_PER_FINGER = 28   # C(8,2) = 28 max per finger
IMPOSTOR_PER_GENUINE         = 1.0  # 1:1 ratio

def discover_images(data_dir):
    fingers = defaultdict(list)
    search_root = data_dir / "Dbs" if (data_dir / "Dbs").exists() else data_dir
    for db_dir in sorted(search_root.iterdir()):
        if not db_dir.is_dir(): continue
        for f in sorted(db_dir.iterdir()):
            m = FNAME_RE.match(f.name)
            if m:
                finger_id = f"{db_dir.name}_{int(m.group(1)):03d}"
                fingers[finger_id].append(f)
    return fingers

fingers = discover_images(DATA_DIR)
print(f"Found {len(fingers)} unique fingers")
print(f"Sample counts: {sorted(set(len(v) for v in fingers.values()))}")

# ── Finger-level split ────────────────────────────────────────────────────────
rng        = random.Random(SEED)
finger_ids = sorted(fingers.keys())
rng.shuffle(finger_ids)
n       = len(finger_ids)
n_train = int(round(n * 0.70))
n_val   = int(round(n * 0.15))
train_fingers = set(finger_ids[:n_train])
val_fingers   = set(finger_ids[n_train:n_train+n_val])
test_fingers  = set(finger_ids[n_train+n_val:])
print(f"Fingers — train:{len(train_fingers)}  val:{len(val_fingers)}  test:{len(test_fingers)}")

def build_pairs(finger_set, split_name):
    split_fingers = {fid: fingers[fid] for fid in finger_set}
    genuine = []
    for fid, imgs in split_fingers.items():
        combos = list(itertools.combinations(sorted(imgs), 2))
        if len(combos) > MAX_GENUINE_PAIRS_PER_FINGER:
            combos = rng.sample(combos, MAX_GENUINE_PAIRS_PER_FINGER)
        for a, b in combos:
            genuine.append({"image_path_1": str(a.relative_to(BASE_DIR)),
                            "image_path_2": str(b.relative_to(BASE_DIR)),
                            "label": 1, "finger_id_1": fid, "finger_id_2": fid})
    target = int(round(len(genuine) * IMPOSTOR_PER_GENUINE))
    fid_list = sorted(split_fingers.keys())
    seen, impostor, attempts = set(), [], 0
    while len(impostor) < target and attempts < max(50_000, target*30):
        attempts += 1
        f1, f2 = rng.sample(fid_list, 2)
        a = rng.choice(split_fingers[f1]); b = rng.choice(split_fingers[f2])
        key = tuple(sorted((str(a), str(b))))
        if key in seen: continue
        seen.add(key)
        impostor.append({"image_path_1": str(a.relative_to(BASE_DIR)),
                         "image_path_2": str(b.relative_to(BASE_DIR)),
                         "label": 0, "finger_id_1": f1, "finger_id_2": f2})
    pairs = genuine + impostor; rng.shuffle(pairs)
    print(f"  {split_name}: {len(genuine)} genuine + {len(impostor)} impostor = {len(pairs)} total")
    return pairs

print("\nBuilding pairs...")
train_pairs = build_pairs(train_fingers, "train")
val_pairs   = build_pairs(val_fingers,   "val")
test_pairs  = build_pairs(test_fingers,  "test")

def write_csv(pairs, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = ["image_path_1","image_path_2","label","finger_id_1","finger_id_2"]
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(pairs)

write_csv(train_pairs, ARTIFACT_DIR/"pairs_train.csv")
write_csv(val_pairs,   ARTIFACT_DIR/"pairs_val.csv")
write_csv(test_pairs,  ARTIFACT_DIR/"pairs_test.csv")
print(f"\n✓ CSVs written to {ARTIFACT_DIR}")


Found 440 unique fingers
Sample counts: [8]
Fingers — train:308  val:66  test:66

Building pairs...
  train: 8624 genuine + 8624 impostor = 17248 total
  val: 1848 genuine + 1848 impostor = 3696 total
  test: 1848 genuine + 1848 impostor = 3696 total

✓ CSVs written to /content/siamvitrm/artifacts


## Dataset and Metrics Helpers

In [9]:
def load_pairs_csv(path):
    with open(path) as f: return list(csv.DictReader(f))

def load_gray(path, size):
    with Image.open(path) as img:
        return np.asarray(img.convert("L").resize(size), dtype=np.float32) / 255.0

# def augment_fingerprint(img: torch.Tensor) -> torch.Tensor:
#     import torchvision.transforms.functional as TF
#     angle  = random.uniform(-12, 12)
#     img    = TF.rotate(img, angle, fill=0.0)
#     factor = 1.0 + random.uniform(-0.2, 0.2)
#     return (img * factor).clamp(0.0, 1.0)

def augment_fingerprint(img: torch.Tensor) -> torch.Tensor:
    import torchvision.transforms.functional as TF
    # Rotation — simulates finger placement angle variation
    angle = random.uniform(-15, 15)
    img = TF.rotate(img, angle, fill=0.0)
    # Translation — simulates off-center placement on sensor
    tx = random.randint(-8, 8)
    ty = random.randint(-8, 8)
    img = TF.affine(img, angle=0, translate=[tx, ty], scale=1.0, shear=0, fill=0.0)
    # Brightness — simulates finger pressure variation
    factor = 1.0 + random.uniform(-0.25, 0.25)
    return (img * factor).clamp(0.0, 1.0)

class FingerprintPairDataset(Dataset):
    def __init__(self, pairs_csv, base_dir, size, max_pairs=None, augment=False):
        self.base_dir = Path(base_dir); self.size = size; self.augment = augment
        self.pairs = load_pairs_csv(pairs_csv)
        if max_pairs and len(self.pairs) > max_pairs:
            random.shuffle(self.pairs); self.pairs = self.pairs[:max_pairs]
        self._cache = {}; self._preload()

    def _preload(self):
        uniq = set()
        for p in self.pairs: uniq.update([p["image_path_1"], p["image_path_2"]])
        h, w = self.size
        print(f"  Preloading {len(uniq)} images at {w}x{h}...")
        for i, rel in enumerate(sorted(uniq)):
            arr = load_gray(self.base_dir / rel, self.size)
            self._cache[rel] = torch.from_numpy(arr).unsqueeze(0)
            if (i+1) % 500 == 0: print(f"    {i+1}/{len(uniq)}")
        print(f"  Cache: {len(self._cache)} images, ~{len(self._cache)*h*w*4/1e6:.0f} MB")

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        import torchvision.transforms.functional as TF
        p  = self.pairs[idx]
        t1 = self._cache[p["image_path_1"]].clone()
        t2 = self._cache[p["image_path_2"]].clone()
        if self.augment:
            t1 = augment_fingerprint(t1); t2 = augment_fingerprint(t2)
            # Same flip for both images in the pair
            if random.random() > 0.5:
                t1 = TF.hflip(t1); t2 = TF.hflip(t2)
        return t1, t2, int(p["label"]), p.get("finger_id_1",""), p.get("finger_id_2","")


def compute_verification_metrics(labels, scores, num_thresholds=500):
    thresholds = np.linspace(float(scores.min()), float(scores.max()), num_thresholds)
    genuine = labels == 1; impostor = labels == 0
    n_gen = int(genuine.sum()); n_imp = int(impostor.sum())
    fars, frrs = [], []
    for t in thresholds:
        pred = scores >= t
        fars.append(float((pred & impostor).sum() / max(n_imp,1)))
        frrs.append(float((~pred & genuine).sum() / max(n_gen,1)))
    fars = np.array(fars); frrs = np.array(frrs)
    eer_idx  = int(np.argmin(np.abs(fars - frrs)))
    eer      = float((fars[eer_idx] + frrs[eer_idx]) / 2)
    thresh   = float(thresholds[eer_idx])
    acc      = float(accuracy_score(labels, (scores >= thresh).astype(int)))
    return {"eer": eer, "eer_threshold": thresh, "accuracy_at_eer": acc,
            "far_at_eer": float(fars[eer_idx]), "frr_at_eer": float(frrs[eer_idx]),
            "thresholds": thresholds, "fars": fars, "frrs": frrs}

def plot_far_frr(metrics, title, save_path=None):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(metrics["thresholds"], metrics["fars"], label="FAR", color="red")
    ax.plot(metrics["thresholds"], metrics["frrs"], label="FRR", color="blue")
    ax.axvline(metrics["eer_threshold"], ls="--", color="gray",
               label=f'EER={metrics["eer"]:.4f} @ τ={metrics["eer_threshold"]:.3f}')
    ax.set_xlabel("Threshold"); ax.set_ylabel("Error rate"); ax.set_title(title); ax.legend()
    fig.tight_layout()
    if save_path: fig.savefig(save_path); print("Saved:", Path(save_path).name)
    plt.show(); return fig

print("✓ Dataset and metrics helpers ready")


✓ Dataset and metrics helpers ready


## Instantiate Model

In [11]:
model = SiamViTRM(
    img_size       = 128,
    patch_size     = 16,
    d_model        = 192,
    n_heads        = 6,
    n_blocks       = 4,
    K              = 24,
    n_latent_steps = 3,    # z-update steps per recursion cycle
    T_recursion    = 1,    # full gradient recursion
).to(DEVICE)

total_params = count_parameters(model)
print(f"SiamViTRM total trainable parameters: {total_params:,}")
print(f"  vs Siamese ResNet-18 baseline:       11,242,176")
print(f"  Parameter reduction factor:          {11_242_176 / total_params:.1f}x")
print()
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20} {n:>8,} params")

# Sanity check
model.eval()
with torch.no_grad():
    d1 = torch.randn(4,1,128,128).to(DEVICE)
    d2 = torch.randn(4,1,128,128).to(DEVICE)
    score, q, y1d, z1d, y2d, z2d, v1, v2 = model(d1, d2)
print(f"\nForward pass OK — score: {score.shape}  range [{score.min():.3f}, {score.max():.3f}]")
model.train()


SiamViTRM total trainable parameters: 1,837,440
  vs Siamese ResNet-18 baseline:       11,242,176
  Parameter reduction factor:          6.1x

  patch_embed            61,440 params
  encoder              1,775,808 params
  halt_head                 192 params

Forward pass OK — score: torch.Size([4])  range [0.432, 0.614]


SiamViTRM(
  (patch_embed): FingerprintPatchEmbed(
    (proj): Linear(in_features=256, out_features=192, bias=False)
  )
  (encoder): ViTRMEncoder(
    (block): Sequential(
      (0): TRMLayer(
        (norm1): RMSNorm()
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=192, out_features=192, bias=False)
        )
        (norm2): RMSNorm()
        (ffn): SwiGLU(
          (w1): Linear(in_features=192, out_features=512, bias=False)
          (w2): Linear(in_features=512, out_features=192, bias=False)
          (w3): Linear(in_features=192, out_features=512, bias=False)
        )
      )
      (1): TRMLayer(
        (norm1): RMSNorm()
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=192, out_features=192, bias=False)
        )
        (norm2): RMSNorm()
        (ffn): SwiGLU(
          (w1): Linear(in_features=192, out_features=512, bias=False)
          (w2): Linear(in_features=512,

## Training Setup

In [12]:
# ── Hyperparameters (matching local notebook exactly) ─────────────────────────
TRAIN_PAIRS_CAP = None
VAL_PAIRS_CAP   = None
TEST_PAIRS_CAP  = 2_500
BATCH_SIZE      = 32
EPOCHS          = 20
LR              = 3e-4
WEIGHT_DECAY    = 0.05
N_SUPERVISION   = 1
EMA_DECAY       = 0.995
GRAD_CLIP       = 1.0

# ── Loss function: BCE (simple, consistent, matches local notebook) ────────────
# def compute_step_loss(score: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
#     return F.binary_cross_entropy(score, labels.float())

def compute_step_loss(score: torch.Tensor, labels: torch.Tensor,
                      v1: torch.Tensor, v2: torch.Tensor) -> torch.Tensor:
    bce = F.binary_cross_entropy(score, labels.float())
    lbl_signed = labels.float() * 2.0 - 1.0   # {0,1} → {-1,+1}
    cel = F.cosine_embedding_loss(v1, v2, lbl_signed, margin=0.3)
    return bce + 0.5 * cel

print("Loss function: BCE(score, label)")

# ── Datasets ──────────────────────────────────────────────────────────────────
print("\nLoading train set...")
train_ds = FingerprintPairDataset(ARTIFACT_DIR/"pairs_train.csv", BASE_DIR,
                                   IMG_SIZE_TRM, max_pairs=TRAIN_PAIRS_CAP, augment=True)
print("Loading val set...")
val_ds   = FingerprintPairDataset(ARTIFACT_DIR/"pairs_val.csv",   BASE_DIR,
                                   IMG_SIZE_TRM, max_pairs=VAL_PAIRS_CAP, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ── Optimizer ─────────────────────────────────────────────────────────────────
optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = max(1, int(0.10 * total_steps))
MIN_LR_FRAC  = 0.05

def get_lr_scale(step):
    if step < warmup_steps: return step / warmup_steps
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return MIN_LR_FRAC + (1 - MIN_LR_FRAC) * 0.5 * (1 + math.cos(math.pi * p))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_scale)
ema        = EMA(model, decay=EMA_DECAY)

print(f"\nTrain pairs: {len(train_ds)},  Val: {len(val_ds)}")
print(f"Batches/epoch: {len(train_loader)},  Total steps: {total_steps},  Warmup: {warmup_steps}")


Loss function: BCE(score, label)

Loading train set...
  Preloading 2464 images at 128x128...
    500/2464
    1000/2464
    1500/2464
    2000/2464
  Cache: 2464 images, ~161 MB
Loading val set...
  Preloading 528 images at 128x128...
    500/528
  Cache: 528 images, ~35 MB

Train pairs: 17248,  Val: 3696
Batches/epoch: 539,  Total steps: 10780,  Warmup: 1078


## Training Loop

In [13]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

train_losses, val_losses = [], []
global_step  = 0
best_val_loss  = float("inf")
best_val_eer   = float("inf")   # add this
best_ema_state = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    epoch_loss = 0.0
    n_batches  = len(train_loader)

    for bi, (img1, img2, labels, _, _) in enumerate(train_loader):
        img1   = img1.to(DEVICE)
        img2   = img2.to(DEVICE)
        labels = labels.to(DEVICE)

        y1, z1, y2, z2 = None, None, None, None
        batch_loss = 0.0

        for sup_step in range(N_SUPERVISION):
            score, q, y1, z1, y2, z2, v1, v2 = model(img1, img2, y1, z1, y2, z2)
            step_loss  = compute_step_loss(score, labels, v1, v2)
            batch_loss += step_loss.item()
            optimizer.zero_grad()
            step_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            ema.update(model)
            global_step += 1

        epoch_loss += batch_loss / N_SUPERVISION
        if (bi + 1) % 50 == 0:
            print(f"  Epoch {epoch} batch {bi+1}/{n_batches} "
                  f"loss={batch_loss/N_SUPERVISION:.4f} lr={optimizer.param_groups[0]['lr']:.2e}")

    train_losses.append(epoch_loss / n_batches)

    # ── Validation (BCE on EMA model, same as local notebook) ────────────────
    ema.eval_model.eval()
    vloss = 0.0
    val_scores_list, val_labels_list = [], []      # add this line
    with torch.no_grad():
        for img1, img2, labels, _, _ in val_loader:
            img1 = img1.to(DEVICE); img2 = img2.to(DEVICE); labels = labels.to(DEVICE)
            score, q, *_ = ema.eval_model(img1, img2)
            vloss += F.binary_cross_entropy(score, labels.float()).item()
            val_scores_list.append(score.cpu().numpy())    # add this line
            val_labels_list.append(labels.cpu().numpy())         # add this line
    val_losses.append(vloss / len(val_loader))

    # Compute val EER
    val_scores_arr = np.concatenate(val_scores_list)
    val_labels_arr = np.concatenate(val_labels_list)
    val_eer = compute_verification_metrics(val_labels_arr, val_scores_arr)["eer"]

    # if val_losses[-1] < best_val_loss:
    #     best_val_loss  = val_losses[-1]
    #     best_ema_state = copy.deepcopy(ema.eval_model.state_dict())
    #     torch.save(best_ema_state, TRM_DIR / "siamvitrm_ema_best.pt")
    #     print(f"  ✓ New best val loss: {best_val_loss:.4f} — saving EMA checkpoint")

    if val_eer < best_val_eer:
      best_val_eer   = val_eer
      best_ema_state = copy.deepcopy(ema.eval_model.state_dict())
      torch.save(best_ema_state, TRM_DIR / "siamvitrm_ema_best.pt")
      print(f"  ✓ New best val EER: {best_val_eer*100:.2f}% — saving EMA checkpoint")

    print(f"Epoch {epoch}/{EPOCHS}  train_loss={train_losses[-1]:.4f}  "
          f"val_loss={val_losses[-1]:.4f}  val_eer={val_eer*100:.2f}%  ({time.time()-t0:.1f}s)")

if best_ema_state is not None:
    ema.eval_model.load_state_dict(best_ema_state)
    # print(f"\nRestored best EMA checkpoint (val_loss={best_val_loss:.4f})")
    print(f"\nRestored best EMA checkpoint (val_eer={best_val_eer*100:.2f}%)")

torch.save(model.state_dict(),          TRM_DIR / "siamvitrm_main.pt")
torch.save(ema.eval_model.state_dict(), TRM_DIR / "siamvitrm_ema.pt")
print("✓ Checkpoints saved")


  Epoch 1 batch 50/539 loss=0.8064 lr=1.39e-05
  Epoch 1 batch 100/539 loss=0.9897 lr=2.78e-05
  Epoch 1 batch 150/539 loss=0.7175 lr=4.17e-05
  Epoch 1 batch 200/539 loss=0.8469 lr=5.57e-05
  Epoch 1 batch 250/539 loss=0.7847 lr=6.96e-05
  Epoch 1 batch 300/539 loss=0.7713 lr=8.35e-05
  Epoch 1 batch 350/539 loss=0.6244 lr=9.74e-05
  Epoch 1 batch 400/539 loss=0.8244 lr=1.11e-04
  Epoch 1 batch 450/539 loss=0.8913 lr=1.25e-04
  Epoch 1 batch 500/539 loss=1.1861 lr=1.39e-04
  ✓ New best val EER: 21.78% — saving EMA checkpoint
Epoch 1/20  train_loss=0.9489  val_loss=0.8033  val_eer=21.78%  (108.9s)
  Epoch 2 batch 50/539 loss=1.0944 lr=1.64e-04
  Epoch 2 batch 100/539 loss=0.8903 lr=1.78e-04
  Epoch 2 batch 150/539 loss=0.7723 lr=1.92e-04
  Epoch 2 batch 200/539 loss=0.8696 lr=2.06e-04
  Epoch 2 batch 250/539 loss=0.6479 lr=2.20e-04
  Epoch 2 batch 300/539 loss=0.6746 lr=2.33e-04
  Epoch 2 batch 350/539 loss=0.7388 lr=2.47e-04
  Epoch 2 batch 400/539 loss=0.9324 lr=2.61e-04
  Epoch 2 ba

In [14]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(train_losses)+1), train_losses, marker="o", label="Train",     color="steelblue")
ax.plot(range(1, len(val_losses)+1),   val_losses,   marker="s", label="Val (EMA)", color="darkorange")
ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
ax.set_title("SiamViTRM — Training and Validation Loss"); ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "siamvitrm_training_curve.png"); plt.show()
print("✓ Saved training curve")


✓ Saved training curve


## Evaluation on Test Set

In [15]:
print("Loading test set...")
test_ds     = FingerprintPairDataset(ARTIFACT_DIR/"pairs_test.csv", BASE_DIR,
                                     IMG_SIZE_TRM, max_pairs=None)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

ema.eval_model.eval()
all_scores, all_labels = [], []
with torch.no_grad():
    for img1, img2, labels, _, _ in test_loader:
        img1 = img1.to(DEVICE); img2 = img2.to(DEVICE)
        score, _, *_ = ema.eval_model(img1, img2)
        all_scores.append(score.cpu().numpy()); all_labels.append(labels.numpy())

trm_scores  = np.concatenate(all_scores)
trm_labels  = np.concatenate(all_labels)
print(f"Evaluated {len(trm_scores)} test pairs")
print(f"Score range: [{trm_scores.min():.4f}, {trm_scores.max():.4f}]")

trm_metrics = compute_verification_metrics(trm_labels, trm_scores)
print("=" * 50)
print("SiamViTRM Test Results")
print("=" * 50)
print(f"  EER:          {trm_metrics['eer']:.4f}  ({trm_metrics['eer']*100:.2f}%)")
print(f"  Accuracy:     {trm_metrics['accuracy_at_eer']:.4f}")
print(f"  FAR @ EER:    {trm_metrics['far_at_eer']:.4f}")
print(f"  FRR @ EER:    {trm_metrics['frr_at_eer']:.4f}")
print(f"  Threshold:    {trm_metrics['eer_threshold']:.4f}")


Loading test set...
  Preloading 528 images at 128x128...
    500/528
  Cache: 528 images, ~35 MB
Evaluated 3696 test pairs
Score range: [0.1694, 0.9999]
SiamViTRM Test Results
  EER:          0.1588  (15.88%)
  Accuracy:     0.8412
  FAR @ EER:    0.1596
  FRR @ EER:    0.1580
  Threshold:    0.8734


In [16]:
plot_far_frr(trm_metrics, "FAR vs FRR — SiamViTRM (FVC2004)",
             FIG_DIR / "far_frr_siamvitrm.png")

genuine_scores  = trm_scores[trm_labels == 1]
impostor_scores = trm_scores[trm_labels == 0]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(genuine_scores,  bins=50, alpha=0.7, label="Genuine",  color="steelblue",  density=True)
ax.hist(impostor_scores, bins=50, alpha=0.7, label="Impostor", color="darkorange", density=True)
ax.axvline(trm_metrics["eer_threshold"], ls="--", color="gray",
           label=f'EER threshold = {trm_metrics["eer_threshold"]:.3f}')
ax.set_xlabel("Verification Score"); ax.set_ylabel("Density")
ax.set_title("SiamViTRM — Score Distribution (Genuine vs Impostor)"); ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "siamvitrm_score_distribution.png"); plt.show()
print("✓ Plots saved")


Saved: far_frr_siamvitrm.png
✓ Plots saved


## Error Analysis

In [17]:
threshold = trm_metrics["eer_threshold"]
predicted = (trm_scores >= threshold).astype(int)
eval_pairs = test_ds.pairs[:len(trm_labels)]

false_accepts, false_rejects = [], []
for i, p in enumerate(eval_pairs):
    true_lbl = int(p["label"]); pred_lbl = int(predicted[i])
    if pred_lbl == 1 and true_lbl == 0: false_accepts.append(p)
    elif pred_lbl == 0 and true_lbl == 1: false_rejects.append(p)

print(f"False accepts (impostors misclassified as genuine): {len(false_accepts)}")
print(f"False rejects (genuines misclassified as impostors): {len(false_rejects)}")
print(f"Total errors: {len(false_accepts)+len(false_rejects)} / {len(trm_labels)}")
print(f"Error rate:   {(len(false_accepts)+len(false_rejects))/len(trm_labels):.4f}")

def get_partition(finger_id):
    parts = finger_id.rsplit("_", 1)
    return parts[0] if len(parts) == 2 else finger_id

cross_fa = sum(1 for p in false_accepts
               if get_partition(p.get("finger_id_1","")) != get_partition(p.get("finger_id_2","")))
cross_fr = sum(1 for p in false_rejects
               if get_partition(p.get("finger_id_1","")) != get_partition(p.get("finger_id_2","")))
print(f"\nCross-partition false accepts: {cross_fa}/{len(false_accepts)} "
      f"({100*cross_fa/max(len(false_accepts),1):.1f}%)")
print(f"Cross-partition false rejects: {cross_fr}/{len(false_rejects)} "
      f"({100*cross_fr/max(len(false_rejects),1):.1f}%)")


False accepts (impostors misclassified as genuine): 295
False rejects (genuines misclassified as impostors): 292
Total errors: 587 / 3696
Error rate:   0.1588

Cross-partition false accepts: 60/295 (20.3%)
Cross-partition false rejects: 0/292 (0.0%)


## Save Results to Google Drive

In [18]:
trm_results = {
    "model": "SiamViTRM",
    "architecture": {
        "img_size": 128, "patch_size": 16, "n_patches": 64,
        "d_model": 192, "n_heads": 6, "n_blocks": 4,
        "K": 24, "n_latent_steps": 3, "T_recursion": 1,
        "effective_depth": 1 * (3+1) * 4,
    },
    "training": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
        "weight_decay": WEIGHT_DECAY, "N_supervision": N_SUPERVISION,
        "EMA_decay": EMA_DECAY, "train_pairs": len(train_ds),
        "loss_function": "BCE", "pretrained": False,
    },
    "evaluation": {
        "test_pairs": int(len(trm_scores)),
        "eer": float(trm_metrics["eer"]),
        "eer_threshold": float(trm_metrics["eer_threshold"]),
        "accuracy_at_eer": float(trm_metrics["accuracy_at_eer"]),
        "far_at_eer": float(trm_metrics["far_at_eer"]),
        "frr_at_eer": float(trm_metrics["frr_at_eer"]),
        "model_params": int(count_parameters(model)),
        "false_accepts": len(false_accepts),
        "false_rejects": len(false_rejects),
    },
}
with open(TRM_DIR / "trm_results.json", "w") as f:
    json.dump(trm_results, f, indent=2)
print(json.dumps(trm_results, indent=2))

# ── Copy to Google Drive ───────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    import shutil
    drive_out = Path("/content/drive/MyDrive/cs228_siamvitrm_outputs")
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(TRM_DIR), str(drive_out/"trm"), dirs_exist_ok=True)
    print(f"\n✓ Outputs copied to Google Drive: {drive_out}")
except Exception as e:
    print(f"Drive copy skipped ({e}) — files at {TRM_DIR}")


{
  "model": "SiamViTRM",
  "architecture": {
    "img_size": 128,
    "patch_size": 16,
    "n_patches": 64,
    "d_model": 192,
    "n_heads": 6,
    "n_blocks": 4,
    "K": 24,
    "n_latent_steps": 3,
    "T_recursion": 1,
    "effective_depth": 16
  },
  "training": {
    "epochs": 20,
    "batch_size": 32,
    "lr": 0.0003,
    "weight_decay": 0.05,
    "N_supervision": 1,
    "EMA_decay": 0.995,
    "train_pairs": 17248,
    "loss_function": "BCE",
    "pretrained": false
  },
  "evaluation": {
    "test_pairs": 3696,
    "eer": 0.1588203463203463,
    "eer_threshold": 0.8734336293532041,
    "accuracy_at_eer": 0.8411796536796536,
    "far_at_eer": 0.15963203463203463,
    "frr_at_eer": 0.15800865800865802,
    "model_params": 1837440,
    "false_accepts": 295,
    "false_rejects": 292
  }
}
Mounted at /content/drive

✓ Outputs copied to Google Drive: /content/drive/MyDrive/cs228_siamvitrm_outputs
